# Databricks Vector Search 인덱스 생성 및 관리 가이드

Databricks의 Vector Search 기능을 활용하여 테이블 데이터를 벡터화하고 검색 가능한 인덱스를 생성하는 과정을 설명합니다.

인덱스는 Databricks UI 또는 파이썬 라이브러리(`databricks-vectorsearch`)를 통해 생성할 수 있습니다.

관련 문서 : 
- https://learn.microsoft.com/ko-kr/azure/databricks/vector-search/create-vector-search
- databricks-vectorsearch client : https://api-docs.databricks.com/python/vector-search/databricks.vector_search.html


---

## 1. 환경 설정 및 라이브러리 설치

Vector Search 클라이언트를 사용하기 위해 필요한 라이브러리를 설치하고 파이썬 세션을 재시작합니다.

In [0]:
# 벡터 검색 라이브러리 설치
%pip install databricks-vectorsearch 

# 설치 후 라이브러리 업데이트를 위해 파이썬 세션 재시작
dbutils.library.restartPython()

Note: you may need to restart the kernel using %restart_python or dbutils.library.restartPython() to use updated packages.


---

## 2. 클라이언트 초기화 및 인덱스 확인

`VectorSearchClient`를 인스턴스화하여 특정 엔드포인트에 생성된 기존 인덱스 목록을 확인합니다.

이전에, vector search index의 endpoint를 생성해야 합니다.
<table>
  <tr>
    <td style="border:1px solid #ccc; padding:8px;">
      1. Compute -> Vector Search -> Create endpoint
    </td>
  </tr>
  <tr> 
    <td style="border:1px solid #ccc; padding:8px;">
      <img src="/Workspace/Shared/5강_Databricks_AI_Agent_활용/img/make_index_endpoint_01.png" width="800" />
    </td>
  </tr>
</table>
<table>
  <tr>
    <td style="border:1px solid #ccc; padding:8px;">
      1. endpoint 이름 지정 및 모드 선택 후 확인
    </td>
  </tr>
  <tr>
    <td style="border:1px solid #ccc; padding:8px;">
      <img src="/Workspace/Shared/5강_Databricks_AI_Agent_활용/img/make_index_endpoint_02.png" width="800" />
    </td>
  </tr>
</table>


In [0]:
from databricks.vector_search.client import VectorSearchClient

client = VectorSearchClient()

# 지정된 벡터 검색 엔드포인트 내의 모든 인덱스 리스트 출력
indexes = client.list_indexes("vector_search_endpoint")
print(indexes)

[NOTICE] Using a notebook authentication token. Recommended for development only. For improved performance, please use Service Principal based authentication. To disable this message, pass disable_notice=True.
{}


---

## 3. Vector Search Index 생성

Delta 테이블의 데이터가 변경될 때마다 자동으로(또는 트리거 시) 동기화되는 벡터 인덱스를 생성합니다. <br>이 과정에서 텍스트 데이터는 임베딩 모델을 통해 벡터로 변환됩니다.<br> 

이전에 소스로 사용될 테이블에서 CDF 기능을 활성화 해야 합니다.
> CDF 기능 : 델타 테이블 버전 간의 행 수준 변경 사항을 추적<br>
> https://docs.databricks.com/aws/en/delta/delta-change-data-feed

---

In [0]:
%sql
ALTER TABLE ddbx_20260120_academy.vector_source.tourism_festival_summary SET TBLPROPERTIES (delta.enableChangeDataFeed = true);

#### 3.1 UI를 통한 생성
<table>
  <tr>
    <td style="border:1px solid #ccc; padding:8px;">
      1. Catalog -> table 선택 -> Create -> Vector search index
    </td>
  </tr>
  <tr>
    <td style="border:1px solid #ccc; padding:8px;">
      <img src="/Workspace/Shared/5강_Databricks_AI_Agent_활용/img/make_index_01.png" width="800" />
    </td>
  </tr>
</table>
<table>
  <tr>
    <td style="border:1px solid #ccc; padding:8px;">
      2. 위치/이름, Pk , 임베딩 소스로 사용될 컬럼, 임베딩 모델 엔드포인트,<br> 백터 서치 인덱스 엔드포인트, 동기화 모드 지정
    </td>
  </tr>
  <tr>
    <td style="border:1px solid #ccc; padding:8px;">
      <img src="/Workspace/Shared/5강_Databricks_AI_Agent_활용/img/make_index_02.png" width="400" />
    </td>
  </tr>
</table>


---
#### 3.2 python 라이브러리(databricks-vectorsearch)를 통한 생성

In [0]:
index = client.create_delta_sync_index( 
  # 벡터 검색 엔드포인트 이름                                     
  endpoint_name="vector_search_endpoint",

  # 소스 데이터가 되는 테이블 경로 (Catalog.Schema.Table)
  source_table_name="ddbx_20260120_academy.vector_source.tourism_festival_summary",

  # 생성될 벡터 인덱스의 고유 이름 (Catalog.Schema.Table)
  index_name="ddbx_20260120_academy.vector_index.tourism_festival_summary_index",

  # 동기화 방식: 'TRIGGERED'(수동/스케줄 실행) / 'CONTINUOUS'(실시간 동기화)
  # -> TRIGGERED는 필요할 때만 업데이트하여 비용을 절감하기 좋습니다.
  pipeline_type="TRIGGERED",

  # 소스 테이블의 기본 키(Primary Key) 컬럼
  primary_key="id",

  # 벡터화할 텍스트 데이터가 포함된 컬럼
  embedding_source_column="description",

  # 임베딩을 수행할 모델 서비스 엔드포인트 이름
  embedding_model_endpoint_name="bge_m3_service", 
)

#### 3.3 Troubleshooting : TEMPORARILY_UNAVAILABLE
- 임베딩 모델의 서빙 엔드포인트의 scale이 zero일 경우 아래와 같은 오류가 발생됩니다.

```
Exception: Response content b'{
  "error_code":"TEMPORARILY_UNAVAILABLE",
  "message":"Model serving endpoint bge-m3-ext timed out after 60s. 
    Please check the model serving endpoint status. 
    Error: grpc_shaded.com.linecorp.armeria.client.ResponseTimeoutException",
  "details":[{"@type":"type.googleapis.com/google.rpc.RequestInfo",
    "request_id":"3b86aca3-eb81-4738-99f9-de3d6d9b8043",
    "serving_data":""}]}', 
    status_code 503 ...
```
<table>
  <tr>
    <td style="border:1px solid #ccc; padding:8px;">
      [사용 가능 상태] READY 상태일 때 실행이 가능합니다.
    </td>
  </tr>
  <tr>
    <td style="border:1px solid #ccc; padding:8px;">
      <img src="/Workspace/Shared/5강_Databricks_AI_Agent_활용/img/embedding_ready_state.png" width="800" />
    </td>
  </tr>
</table>

<table>
  <tr>
    <td style="border:1px solid #ccc; padding:8px;">
      [사용 불가 상태] READY(Scaled to zero) 상태일 때 실행이 불가합니다.
      컴퓨팅 자원(CPU, GPU, RAM)을 0개로 완전히 줄여버리는 상태를 의미합니다.
    </td>
  </tr>
  <tr>
    <td style="border:1px solid #ccc; padding:8px;">
      <img src="/Workspace/Shared/5강_Databricks_AI_Agent_활용/img/embedding_ready_state2.png" width="800" />
    </td>
  </tr>
</table>

#### 3.4 Troubleshooting : BAD_REQUEST
- 권한 문제 또는 embedding model 의 스펙이 모자라는 경우 발생합니다.
```
java.lang.Exception: 
Error: Response Code: 400, 
Response: {"error_code":"BAD_REQUEST",
  "message":"invalid GetInternalCredentialsForIndexRequest!",
  "details":[
      {"@type":"type.googleapis.com/google.rpc.RequestInfo",
    "request_id":"9b464034-db67-41af-ba42-a105d25c2f4f","serving_data":""}]
}
```

---

## 4. 진행 확인
<table>
  <tr>
    <td style="border:1px solid #ccc; padding:8px;">
      <img src="/Workspace/Shared/5강_Databricks_AI_Agent_활용/img/make_index_03.png" width="500" />
    </td>
        <td style="border:1px solid #ccc; padding:8px;">
      <img src="/Workspace/Shared/5강_Databricks_AI_Agent_활용/img/make_index_04.png" width="500" />
    </td>
  </tr>
</table>

---

## 5. 인덱스 동기화

`index.sync()` 메서드는 원본 테이블의 변경 사항을 벡터 인덱스에 즉시 반영하기 위해 사용됩니다.<br>
인덱스 생성 시 `pipeline_type="TRIGGERED"`로 설정된 경우에만 사용 가능합니다. (`CONTINUOUS` 모드는 자동으로 실시간 동기화가 일어나므로 이 명령어가 필요하지 않습니다.)

- trigger option : https://api-docs.databricks.com/python/vector-search/databricks.vector_search.html#databricks.vector_search.index.VectorSearchIndex.sync

In [0]:
client = VectorSearchClient()
index = client.get_index(index_name="ddbx_20260120_academy.vector_index.tourism_festival_summary_index")

# 원본 테이블과 인덱스 간의 동기화 시작
index.sync()

<table>
  <tr>
    <td style="border:1px solid #ccc; padding:8px;">
      <img src="/Workspace/Shared/5강_Databricks_AI_Agent_활용/img/sync.png" width="800" />
    </td>
    <td style="border:1px solid #ccc; padding:8px;">
      <img src="/Workspace/Shared/5강_Databricks_AI_Agent_활용/img/sync2.png" width="800" />
    </td>
  </tr>
  <tr>
    <td style="border:1px solid #ccc; padding:8px;">
    trigger 동기화 중
    </td>
    <td style="border:1px solid #ccc; padding:8px;">
    trigger 동기화 후
    </td>
  </tr>
</table>